# EDA Pulso TransMi

Este notebook documenta la primera lectura analitica del reto. La idea es responder, con evidencia reproducible, tres preguntas antes de entrenar modelos: si los datos son confiables, que patrones temporales/geograficos dominan la demanda y que tan fuertes son baselines simples.

## 1. Reproducibilidad

Antes de abrir este notebook, ejecuta desde la raiz del repositorio:

```bash
python -m src.ingest
python -m src.data_quality
python -m src.eda
```

Los CSV crudos quedan en `data/raw/` y no se versionan. Las tablas y figuras livianas quedan en `reports/eda/`.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = next(
    path for path in [Path.cwd(), Path.cwd().parent]
    if (path / 'pyproject.toml').exists() and (path / 'README.md').exists()
)

REPORT_DIR = PROJECT_ROOT / 'reports' / 'eda'
TABLE_DIR = REPORT_DIR / 'tables'
FIGURE_DIR = REPORT_DIR / 'figures'

if not (REPORT_DIR / 'data_quality.md').exists():
    raise FileNotFoundError(
        'No existe reports/eda/data_quality.md. Ejecuta primero desde la raiz del repo: '
        'python -m src.ingest && python -m src.data_quality && python -m src.eda'
    )

## 2. Calidad de datos

La calidad se valida antes del analisis: conteos esperados, nulos, duplicados, continuidad de 15 minutos, cobertura por estacion y preservacion de ceros iniciales en `station_id`.

In [ ]:
display(Markdown((REPORT_DIR / 'data_quality.md').read_text()))

## 3. Patrones temporales

La demanda se observa cada 15 minutos. Por eso miramos primero agregados diarios, perfiles por hora y dia de semana. Estos patrones nos dicen si un baseline estacional simple puede ser competitivo.

In [ ]:
display(Image(filename=FIGURE_DIR / '01_demanda_diaria.png'))
display(Image(filename=FIGURE_DIR / '02_perfil_horario.png'))
display(Image(filename=FIGURE_DIR / '03_perfil_semanal.png'))

## 4. Diferencias por estacion

Las estaciones no deben mezclarse sin cuidado: pueden tener volumen, variabilidad y picos distintos. Este bloque identifica estaciones dominantes y estaciones potencialmente dificiles por ruido o picos.

In [ ]:
station_summary = pd.read_csv(TABLE_DIR / 'station_summary.csv')
display(station_summary.head(12))
display(Image(filename=FIGURE_DIR / '04_ranking_estaciones.png'))
display(Image(filename=FIGURE_DIR / '05_heatmap_estacion_hora.png'))
display(Image(filename=FIGURE_DIR / '06_distribucion_estaciones.png'))

## 5. Contexto: clima y eventos

El contexto puede explicar parte de la variacion residual. Aqui revisamos distribuciones y cruces simples; no asumimos causalidad todavia, solo buscamos senales utiles para features posteriores.

In [ ]:
display(pd.read_csv(TABLE_DIR / 'context_summary.csv'))
display(Image(filename=FIGURE_DIR / '07_distribucion_lluvia.png'))
display(Image(filename=FIGURE_DIR / '08_distribucion_temperatura.png'))
display(Image(filename=FIGURE_DIR / '09_demanda_vs_lluvia.png'))
display(Image(filename=FIGURE_DIR / '10_demanda_contexto.png'))

## 6. Validacion temporal y baselines

Usamos los ultimos 7 dias como validacion. Un split aleatorio mezclaria pasado y futuro, inflaria metricas y no representaria la operacion real del reto. Comparamos dos baselines: rezago de 24 horas y promedio historico por estacion, dia de semana y bloque de 15 minutos.

In [ ]:
baseline_summary = pd.read_csv(TABLE_DIR / 'baseline_summary.csv')
baseline_station_scores = pd.read_csv(TABLE_DIR / 'baseline_station_scores.csv')
display(baseline_summary)
display(baseline_station_scores.sort_values(['baseline', 'accuracy']).head(24))
display(Image(filename=FIGURE_DIR / '11_accuracy_baselines.png'))
display(Image(filename=FIGURE_DIR / '12_accuracy_estacion_baseline.png'))

## 7. Conclusiones operativas

- Si la calidad queda aprobada, podemos usar estos datos como corte inicial confiable.
- Los baselines fijan el piso minimo que cualquier modelo debe superar.
- Las diferencias por estacion justifican evaluar metricas por estacion y no solo promedio global.
- La siguiente etapa debe convertir estos hallazgos en features y backtesting temporal mas formal.